# 🏋️ MediaPipe Pose — Landmark Extraction & Model Training

This notebook explains the **full pipeline** used in our Workout Monitoring System:

1. What is MediaPipe Pose and how it works
2. Extracting body landmarks from video frames
3. Feature engineering (joint angles, distances, velocities)
4. Building a labelled dataset
5. Training a classifier (Random Forest + LSTM)
6. Evaluating the model

---

## 1. What is MediaPipe Pose?

**MediaPipe Pose** is a machine-learning solution by Google that detects **33 body key-points** (landmarks) in an image or video frame in real-time.

### 33 Pose Landmarks:
```
 0: Nose            11: L_Shoulder     23: L_Hip
 1: L_Eye_Inner     12: R_Shoulder     24: R_Hip
 2: L_Eye           13: L_Elbow        25: L_Knee
 3: L_Eye_Outer     14: R_Elbow        26: R_Knee
 4: R_Eye_Inner     15: L_Wrist        27: L_Ankle
 5: R_Eye           16: R_Wrist        28: R_Ankle
 6: R_Eye_Outer     17: L_Pinky        29: L_Heel
 7: L_Ear           18: R_Pinky        30: R_Heel
 8: R_Ear           19: L_Index        31: L_Foot_Index
 9: L_Mouth         20: R_Index        32: R_Foot_Index
10: R_Mouth         21: L_Thumb
                    22: R_Thumb
```

Each landmark has **4 values**: `x`, `y`, `z`, `visibility`
- `x`, `y` — normalized [0,1] position in the frame
- `z` — depth relative to the hip (approximate)
- `visibility` — confidence that the landmark is visible

Total raw features per frame = **33 × 4 = 132 values**

### How does it work internally?
MediaPipe Pose uses a two-step pipeline:
1. **Pose detection** — A lightweight BlazePose detector finds the person ROI
2. **Pose estimation** — A regression network predicts all 33 landmarks

This allows real-time inference (30+ FPS) even on CPU.


## 2. Setup & Imports

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import to_rgba
import warnings
warnings.filterwarnings('ignore')

# MediaPipe components
mp_pose     = mp.solutions.pose
mp_drawing  = mp.solutions.drawing_utils
mp_styles   = mp.solutions.drawing_styles

print("✓ Libraries loaded")
print(f"  MediaPipe version : {mp.__version__}")
print(f"  OpenCV version    : {cv2.__version__}")
print(f"  NumPy version     : {np.__version__}")

## 3. Understanding Landmarks — Diagram

In [ ]:
"""
Draw a synthetic human stick figure with MediaPipe landmark numbering.
This helps understand which index corresponds to which joint.
"""

# Approximate (x, y) positions for 33 landmarks on a stick figure
landmark_pos = {
    0:  (0.50, 0.05),   # Nose
    1:  (0.47, 0.04),   # L_Eye_Inner
    2:  (0.45, 0.04),   # L_Eye
    3:  (0.43, 0.04),   # L_Eye_Outer
    4:  (0.53, 0.04),   # R_Eye_Inner
    5:  (0.55, 0.04),   # R_Eye
    6:  (0.57, 0.04),   # R_Eye_Outer
    7:  (0.43, 0.07),   # L_Ear
    8:  (0.57, 0.07),   # R_Ear
    9:  (0.48, 0.08),   # L_Mouth
    10: (0.52, 0.08),   # R_Mouth
    11: (0.38, 0.22),   # L_Shoulder
    12: (0.62, 0.22),   # R_Shoulder
    13: (0.30, 0.38),   # L_Elbow
    14: (0.70, 0.38),   # R_Elbow
    15: (0.25, 0.52),   # L_Wrist
    16: (0.75, 0.52),   # R_Wrist
    17: (0.22, 0.56),   # L_Pinky
    18: (0.78, 0.56),   # R_Pinky
    19: (0.23, 0.55),   # L_Index
    20: (0.77, 0.55),   # R_Index
    21: (0.24, 0.55),   # L_Thumb
    22: (0.76, 0.55),   # R_Thumb
    23: (0.42, 0.52),   # L_Hip
    24: (0.58, 0.52),   # R_Hip
    25: (0.40, 0.70),   # L_Knee
    26: (0.60, 0.70),   # R_Knee
    27: (0.38, 0.88),   # L_Ankle
    28: (0.62, 0.88),   # R_Ankle
    29: (0.37, 0.93),   # L_Heel
    30: (0.63, 0.93),   # R_Heel
    31: (0.36, 0.97),   # L_Foot_Index
    32: (0.64, 0.97),   # R_Foot_Index
}

connections = [
    (11,12),(11,13),(12,14),(13,15),(14,16),
    (11,23),(12,24),(23,24),
    (23,25),(24,26),(25,27),(26,28),
    (27,29),(28,30),(29,31),(30,32),
    (0,1),(1,2),(2,3),(3,7),(0,4),(4,5),(5,6),(6,8),
]

key_joints = {11,12,13,14,15,16,23,24,25,26,27,28}

fig, ax = plt.subplots(figsize=(8, 11))
ax.set_facecolor('#0f0f1a')
fig.patch.set_facecolor('#0f0f1a')
ax.set_xlim(0, 1)
ax.set_ylim(1, 0)
ax.axis('off')

for a, b in connections:
    x = [landmark_pos[a][0], landmark_pos[b][0]]
    y = [landmark_pos[a][1], landmark_pos[b][1]]
    ax.plot(x, y, color='#0f3460', linewidth=2.5, zorder=1)

for idx, (x, y) in landmark_pos.items():
    color = '#e94560' if idx in key_joints else '#9a9ab0'
    size  = 120 if idx in key_joints else 60
    ax.scatter(x, y, s=size, color=color, zorder=3, edgecolors='white', linewidths=0.5)
    ax.annotate(str(idx), (x, y), textcoords='offset points',
                xytext=(6, 4), fontsize=7, color='white', zorder=4)

red_patch  = mpatches.Patch(color='#e94560', label='Key workout joints (used for angle/rep counting)')
grey_patch = mpatches.Patch(color='#9a9ab0', label='Face / extremity landmarks')
ax.legend(handles=[red_patch, grey_patch], loc='lower center',
          facecolor='#1a1a2e', edgecolor='#0f3460',
          labelcolor='white', fontsize=9)

ax.set_title('MediaPipe Pose — 33 Body Landmarks', color='white', fontsize=14, pad=14)
plt.tight_layout()
plt.show()

## 4. Extracting Raw Landmarks from a Camera Frame

In [ ]:
"""
Capture one frame from the webcam → run MediaPipe → show landmarks.
If no camera is available, we generate a synthetic frame for demo.
"""

def capture_one_frame(camera_id=0):
    cap = cv2.VideoCapture(camera_id)
    ret, frame = cap.read()
    cap.release()
    return frame if ret else None


def run_mediapipe_on_frame(frame):
    """Return (annotated_frame, raw_landmarks_array or None)"""
    with mp_pose.Pose(
        static_image_mode=True,
        model_complexity=1,
        min_detection_confidence=0.5,
    ) as pose:
        rgb     = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(rgb)

        annotated = frame.copy()
        if results.pose_landmarks:
            mp_drawing.draw_landmarks(
                annotated,
                results.pose_landmarks,
                mp_pose.POSE_CONNECTIONS,
                landmark_drawing_spec=mp_styles.get_default_pose_landmarks_style(),
            )

        raw = None
        if results.pose_landmarks:
            raw = []
            for lm in results.pose_landmarks.landmark:
                raw.extend([lm.x, lm.y, lm.z, lm.visibility])
            raw = np.array(raw, dtype=np.float32)

        return annotated, raw, results


# Try to capture from webcam; fall back to a blank canvas
frame = capture_one_frame()
if frame is None:
    print("⚠ No camera found — using blank frame for demo")
    frame = np.full((480, 640, 3), 30, dtype=np.uint8)

annotated, raw_landmarks, results = run_mediapipe_on_frame(frame)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0f0f1a')

axes[0].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
axes[0].set_title('Original Frame', color='white')
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
axes[1].set_title('MediaPipe Pose Overlay', color='white')
axes[1].axis('off')

plt.tight_layout()
plt.show()

if raw_landmarks is not None:
    print(f"\n✓ Landmarks extracted: {raw_landmarks.shape[0]} values")
    print(f"  = 33 landmarks × 4 values (x, y, z, visibility)")
    print(f"\nFirst 5 landmarks (x, y, z, vis):")
    for i in range(5):
        x, y, z, v = raw_landmarks[i*4 : i*4+4]
        print(f"  {mp_pose.PoseLandmark(i).name:20s}: x={x:.3f}  y={y:.3f}  z={z:.3f}  vis={v:.2f}")
else:
    print("⚠ No person detected in frame.")

## 5. Feature Engineering — Why Not Use Raw Landmarks Directly?

Raw `(x, y)` coordinates depend on **where the person stands** in the frame.
Two identical squats from different positions would look different in raw coords.

We engineer **position-invariant features**:

| Feature Type | Description | Why? |
|---|---|---|
| **Joint Angles** | Angle at each joint | Invariant to position & scale |
| **Norm. Distances** | Distance / torso length | Scale invariant |
| **Velocities** | Δ position / Δ time | Captures movement speed |
| **Accelerations** | Δ velocity / Δ time | Rep rhythm detection |

In [ ]:
"""
Feature engineering functions used in the workout monitoring system.
"""

def get_landmark_xy(landmarks_array, idx):
    """Return (x, y) of landmark at given index."""
    return landmarks_array[idx*4 : idx*4+2]


def joint_angle(a, b, c):
    """
    Calculate the angle at joint b (degrees) formed by segments a-b and b-c.
    
    This is the most important feature — it's position and scale invariant.
    
    Example:
        Knee angle = angle(hip, knee, ankle)
        Squat full depth → ~90°
        Standing → ~175°
    """
    a, b, c = np.array(a), np.array(b), np.array(c)
    ba = a - b
    bc = c - b
    cos = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-8)
    return np.degrees(np.arccos(np.clip(cos, -1.0, 1.0)))


def normalized_distance(p1, p2, reference_length):
    """
    Euclidean distance normalized by a reference body length.
    We use shoulder-hip distance as the reference (torso length).
    """
    dist = np.linalg.norm(np.array(p1) - np.array(p2))
    return dist / (reference_length + 1e-8)


def extract_features_from_frame(landmarks_array):
    """
    Extract the engineered feature vector from a single frame's landmarks.
    
    Returns a dict of {feature_name: value}.
    """
    if landmarks_array is None:
        return None

    lm = landmarks_array
    g  = get_landmark_xy    # short alias

    # ── Key points ─────────────────────────────────────────────────────
    nose        = g(lm, 0)
    l_shoulder  = g(lm, 11)
    r_shoulder  = g(lm, 12)
    l_elbow     = g(lm, 13)
    r_elbow     = g(lm, 14)
    l_wrist     = g(lm, 15)
    r_wrist     = g(lm, 16)
    l_hip       = g(lm, 23)
    r_hip       = g(lm, 24)
    l_knee      = g(lm, 25)
    r_knee      = g(lm, 26)
    l_ankle     = g(lm, 27)
    r_ankle     = g(lm, 28)

    # Reference length: avg shoulder-hip distance (torso length)
    torso_len = (
        np.linalg.norm(l_shoulder - l_hip) +
        np.linalg.norm(r_shoulder - r_hip)
    ) / 2

    # ── Joint Angles ───────────────────────────────────────────────────
    features = {}

    features["left_knee_angle"]     = joint_angle(l_hip,    l_knee,    l_ankle)
    features["right_knee_angle"]    = joint_angle(r_hip,    r_knee,    r_ankle)
    features["left_elbow_angle"]    = joint_angle(l_shoulder, l_elbow, l_wrist)
    features["right_elbow_angle"]   = joint_angle(r_shoulder, r_elbow, r_wrist)
    features["left_hip_angle"]      = joint_angle(l_shoulder, l_hip,   l_knee)
    features["right_hip_angle"]     = joint_angle(r_shoulder, r_hip,   r_knee)
    features["left_shoulder_angle"] = joint_angle(l_elbow,  l_shoulder, l_hip)
    features["right_shoulder_angle"]= joint_angle(r_elbow,  r_shoulder, r_hip)

    # ── Normalized Distances ───────────────────────────────────────────
    features["wrist_to_hip_L"]       = normalized_distance(l_wrist,  l_hip,      torso_len)
    features["wrist_to_hip_R"]       = normalized_distance(r_wrist,  r_hip,      torso_len)
    features["knee_to_ankle_L"]      = normalized_distance(l_knee,   l_ankle,    torso_len)
    features["knee_to_ankle_R"]      = normalized_distance(r_knee,   r_ankle,    torso_len)
    features["shoulder_width"]        = normalized_distance(l_shoulder, r_shoulder, torso_len)
    features["hip_width"]             = normalized_distance(l_hip,    r_hip,      torso_len)

    # ── Vertical position ratios ────────────────────────────────────────
    # (tells us if body is upright, prone, etc.)
    features["hip_height"]   = float(np.mean([l_hip[1],    r_hip[1]]))
    features["knee_height"]  = float(np.mean([l_knee[1],   r_knee[1]]))
    features["wrist_height"] = float(np.mean([l_wrist[1],  r_wrist[1]]))

    return features


# Demo: show features from the captured frame
if raw_landmarks is not None:
    features = extract_features_from_frame(raw_landmarks)
    print("Feature vector from current frame:")
    print("=" * 46)
    for name, val in features.items():
        bar = '█' * int(val / 5) if val < 360 else ''
        print(f"  {name:30s}: {val:7.2f}")
    print(f"\nTotal features per frame: {len(features)}")
else:
    print("No landmarks available — showing feature names only:")
    dummy = np.zeros(132)
    features = extract_features_from_frame(dummy)
    for name in features:
        print(f"  {name}")

## 6. Visualizing Key Joint Angles

In [ ]:
"""
Show how different exercises produce different angle distributions.
We simulate angle trajectories for each exercise.
"""

t = np.linspace(0, 4*np.pi, 200)  # 2 complete reps

squat_knee     = 90  + 75 * (0.5 + 0.5 * np.cos(t))
pushup_elbow   = 90  + 65 * (0.5 + 0.5 * np.cos(t))
curl_elbow     = 155 - 100 * (0.5 + 0.5 * np.cos(t))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.patch.set_facecolor('#0f0f1a')

config = [
    (axes[0], squat_knee,   '#e94560', 'Squat — Knee Angle',
     'During squat: full depth ≈ 90°, standing ≈ 170°',
     90, 105, 160),
    (axes[1], pushup_elbow, '#0f3460', 'Push-up — Elbow Angle',
     'Chest to floor: ≈ 90°, fully extended ≈ 160°',
     90, 90, 155),
    (axes[2], curl_elbow,   '#533483', 'Bicep Curl — Elbow Angle',
     'Fully curled ≈ 50°, fully extended ≈ 155°',
     150, 55, 150),
]

for ax, signal, color, title, subtitle, ref, down_t, up_t in config:
    ax.set_facecolor('#1a1a2e')
    ax.plot(signal, color=color, linewidth=2.5, label='Joint Angle')
    ax.axhline(down_t, color='#e94560', linestyle='--', alpha=0.6, label=f'Down thresh {down_t}°')
    ax.axhline(up_t,   color=GOOD_GREEN if True else '#e94560',
               linestyle='--', alpha=0.6, label=f'Up thresh {up_t}°')
    ax.fill_between(range(len(signal)),
                    signal, alpha=0.12, color=color)
    ax.set_title(title, color='white', fontsize=11)
    ax.set_xlabel('Frame', color='#9a9ab0')
    ax.set_ylabel('Angle (°)', color='#9a9ab0')
    ax.tick_params(colors='#9a9ab0')
    ax.spines[:].set_color('#0f3460')
    ax.legend(fontsize=8, loc='upper right',
              facecolor='#0f0f1a', edgecolor='#0f3460', labelcolor='white')
    ax.text(10, signal.min() + 3, subtitle, color='#9a9ab0', fontsize=7.5)

GOOD_GREEN = '#48c774'
plt.suptitle('Joint Angle Trajectories — Rep Counting Signal', color='white', fontsize=13)
plt.tight_layout()
plt.show()

print("""
Rep Counting Logic (State Machine):
────────────────────────────────────────────────────────────
NEUTRAL → angle crosses DOWN threshold → state = DOWN
DOWN    → angle crosses UP threshold   → state = UP  → +1 REP
UP      → angle crosses DOWN threshold → state = DOWN (next rep)
────────────────────────────────────────────────────────────
This is more reliable than peak detection because:
  ✓ No window/buffer required
  ✓ Handles pauses mid-rep
  ✓ Cooldown prevents double-counting
""")

## 7. Building a Dataset

To train a classifier that recognises exercises, we need a labelled dataset.

### Collection Process:
```
Video Frame
    └── MediaPipe Pose
            └── 33 landmarks (x, y, z, vis)
                    └── Feature Engineering
                            └── [angle1, angle2, ..., dist1, dist2, ...]
                                    └── Label (Squat / Push-up / Bicep Curl)
                                            └── Append to CSV
```

Each row = one frame = one feature vector + label.

In [ ]:
"""
Simulate collecting a dataset by generating synthetic landmark trajectories
for each exercise, extracting features, and building a DataFrame.
"""

import random
from ipywidgets import IntProgress

EXERCISES = ["Squat", "Push-up", "Bicep Curl"]
N_FRAMES_PER_EXERCISE = 300


def simulate_landmarks_for_exercise(exercise: str, n_frames: int):
    """
    Generate realistic synthetic landmark arrays for demo purposes.
    In production: run MediaPipe on real videos.
    """
    rows = []
    t = np.linspace(0, 4*np.pi, n_frames)

    for i in range(n_frames):
        # Base landmarks — standing upright
        lm = np.zeros(132)

        # Set base skeleton positions
        coords = {
            11: (0.38, 0.30), 12: (0.62, 0.30),   # shoulders
            13: (0.30, 0.46), 14: (0.70, 0.46),   # elbows
            15: (0.25, 0.60), 16: (0.75, 0.60),   # wrists
            23: (0.42, 0.56), 24: (0.58, 0.56),   # hips
            25: (0.40, 0.75), 26: (0.60, 0.75),   # knees
            27: (0.38, 0.92), 28: (0.62, 0.92),   # ankles
        }

        phase = 0.5 + 0.5 * np.cos(t[i])  # oscillates 0→1→0

        if exercise == "Squat":
            # Knees & hips go down during squat
            drop = phase * 0.15
            coords[25] = (0.40, 0.75 - drop * 0.6)   # knee rises (y decreases = higher)
            coords[26] = (0.60, 0.75 - drop * 0.6)
            coords[23] = (0.42, 0.56 + drop)          # hip drops
            coords[24] = (0.58, 0.56 + drop)

        elif exercise == "Push-up":
            # Elbows bend, shoulders drop
            drop = phase * 0.12
            coords[11] = (0.38, 0.55 + drop)  # shoulders near floor
            coords[12] = (0.62, 0.55 + drop)
            coords[13] = (0.30, 0.60 + drop)
            coords[14] = (0.70, 0.60 + drop)

        elif exercise == "Bicep Curl":
            # Wrists rise up
            rise = phase * 0.28
            coords[15] = (0.32, 0.60 - rise)
            coords[16] = (0.68, 0.60 - rise)
            coords[13] = (0.35, 0.46 - rise * 0.3)
            coords[14] = (0.65, 0.46 - rise * 0.3)

        # Write into landmark array (add small noise)
        for idx, (x, y) in coords.items():
            lm[idx*4]   = x + np.random.normal(0, 0.01)
            lm[idx*4+1] = y + np.random.normal(0, 0.01)
            lm[idx*4+3] = 0.95  # visibility

        features = extract_features_from_frame(lm)
        features["label"] = exercise
        rows.append(features)

    return rows


# Collect dataset
all_rows = []
for ex in EXERCISES:
    rows = simulate_landmarks_for_exercise(ex, N_FRAMES_PER_EXERCISE)
    all_rows.extend(rows)
    print(f"  {ex:15s}: {len(rows)} frames collected")

df = pd.DataFrame(all_rows)
print(f"\n✓ Dataset shape: {df.shape}")
print(f"  {df['label'].value_counts().to_dict()}")
df.head()

## 8. Exploratory Data Analysis — Feature Distributions

In [ ]:
"""
Visualize how features differ between exercises.
Good features = large separation between exercise classes.
"""

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.patch.set_facecolor('#0f0f1a')

colors = {'Squat': '#e94560', 'Push-up': '#4488ff', 'Bicep Curl': '#aa66ff'}

important_features = [
    "left_knee_angle", "left_elbow_angle", "left_hip_angle",
    "wrist_height",    "hip_height",       "shoulder_width",
]

for ax, feat in zip(axes.flat, important_features):
    ax.set_facecolor('#1a1a2e')
    for exercise, color in colors.items():
        data = df[df['label'] == exercise][feat]
        ax.hist(data, bins=30, alpha=0.65, label=exercise,
                color=color, edgecolor='none')
    ax.set_title(feat.replace('_', ' ').title(), color='white', fontsize=10)
    ax.set_xlabel('Value', color='#9a9ab0', fontsize=8)
    ax.tick_params(colors='#9a9ab0')
    ax.spines[:].set_color('#0f3460')
    ax.legend(fontsize=7, facecolor='#0f0f1a', edgecolor='#0f3460', labelcolor='white')

plt.suptitle('Feature Distributions by Exercise Class', color='white', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print("\nObservations:")
print("  • left_knee_angle separates Squats from everything else")
print("  • left_elbow_angle separates Push-ups and Bicep Curls from Squats")
print("  • wrist_height tells us Bicep Curls (wrist goes high)")
print("  • Good features have LOW OVERLAP between exercise classes")

## 9. Model Training — Random Forest Classifier

In [ ]:
"""
Train a Random Forest to classify exercise type from a single frame's features.

Why Random Forest?
  ✓ Handles non-linear decision boundaries
  ✓ Provides feature importance
  ✓ Fast inference (< 1ms)
  ✓ No normalisation required
  ✓ Good baseline before trying deep learning
"""

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Prepare data
feature_cols = [c for c in df.columns if c != 'label']
X = df[feature_cols].values
y = df['label'].values

le = LabelEncoder()
y_enc = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.25, random_state=42, stratify=y_enc
)

print(f"Training samples : {X_train.shape[0]}")
print(f"Test samples     : {X_test.shape[0]}")
print(f"Features per frame: {X_train.shape[1]}")

# Train
rf = RandomForestClassifier(
    n_estimators=150,
    max_depth=10,
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train, y_train)

# Evaluate
y_pred = rf.predict(X_test)
acc = (y_pred == y_test).mean()
print(f"\n✓ Test Accuracy: {acc*100:.1f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0f0f1a')

# --- Confusion matrix ---
axes[0].set_facecolor('#1a1a2e')
im = axes[0].imshow(cm, cmap='Blues')
axes[0].set_xticks(range(len(le.classes_)))
axes[0].set_yticks(range(len(le.classes_)))
axes[0].set_xticklabels(le.classes_, color='white', fontsize=10)
axes[0].set_yticklabels(le.classes_, color='white', fontsize=10)
axes[0].set_xlabel('Predicted', color='#9a9ab0')
axes[0].set_ylabel('Actual',    color='#9a9ab0')
axes[0].set_title('Confusion Matrix', color='white', fontsize=12)

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        axes[0].text(j, i, str(cm[i, j]),
                     ha='center', va='center',
                     color='white', fontsize=14, fontweight='bold')

# --- Feature importance ---
importances = rf.feature_importances_
top_n = 10
top_idx = np.argsort(importances)[-top_n:][::-1]

axes[1].set_facecolor('#1a1a2e')
bars = axes[1].barh(
    [feature_cols[i].replace('_', ' ') for i in top_idx],
    importances[top_idx],
    color='#e94560'
)
axes[1].set_xlabel('Importance', color='#9a9ab0')
axes[1].set_title('Top 10 Feature Importances', color='white', fontsize=12)
axes[1].tick_params(colors='#9a9ab0')
axes[1].spines[:].set_color('#0f3460')

plt.tight_layout()
plt.show()

## 10. LSTM for Temporal Sequence Classification

Random Forest classifies **one frame at a time** — it has no memory.
An **LSTM** (Long Short-Term Memory) can look at a **sequence** of frames and learn movement patterns.

### Why LSTM?

| | Random Forest | LSTM |
|---|---|---|
| Input | 1 frame | Sequence of N frames |
| Temporal context | ❌ None | ✅ Learns patterns over time |
| Training data needed | Less | More |
| Inference speed | Fastest | Fast (< 10ms) |

In [ ]:
"""
Build and explain an LSTM model architecture for exercise classification.

Input shape: (batch, SEQUENCE_LEN, N_FEATURES)
  = (?, 30, 19)  →  30 frames sliding window, 19 features each
"""

try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    print(f"TensorFlow {tf.__version__} available")
    HAS_TF = True
except ImportError:
    print("TensorFlow not installed — showing architecture diagram only")
    HAS_TF = False


SEQ_LEN    = 30   # 1 second @ 30 fps
N_FEATURES = len(feature_cols)
N_CLASSES  = len(EXERCISES)

print(f"""
LSTM Architecture:
──────────────────────────────────────────────────────────
Input       : ({SEQ_LEN}, {N_FEATURES})   ← 30 frames × {N_FEATURES} features
LSTM(128)   : Learns temporal patterns in the sequence
Dropout(0.4): Regularization (prevents overfitting)
LSTM(64)    : Second temporal layer (deeper representation)
Dense(64)   : Non-linear mapping to class space
Dense({N_CLASSES})    : Softmax → probability per exercise
──────────────────────────────────────────────────────────
Output      : ({N_CLASSES},) — [Squat, Push-up, Bicep Curl] probabilities
""")

if HAS_TF:
    model = keras.Sequential([
        layers.Input(shape=(SEQ_LEN, N_FEATURES)),
        layers.LSTM(128, return_sequences=True),
        layers.Dropout(0.4),
        layers.LSTM(64),
        layers.Dense(64, activation='relu'),
        layers.Dense(N_CLASSES, activation='softmax'),
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    model.summary()

    print("\nTo train with real data:")
    print("  history = model.fit(X_train_seq, y_train, epochs=50,")
    print("                      validation_split=0.2, batch_size=32)")

## 11. Putting It All Together — Live Pipeline

In [ ]:
"""
This shows the exact pipeline that runs in app/monitor.py:

 Camera Frame
     │
     ▼
 MediaPipe Pose
     │  33 landmarks (x,y,z,vis)
     ▼
 Feature Engineering
     │  joint angles, distances (19 features)
     ▼
 Exercise Analyzer
     │  State machine: NEUTRAL→DOWN→UP = +1 rep
     │  Form check: angle thresholds → feedback messages
     ▼
 HUD Overlay (draw_hud)
     │  Rep count, form status, messages, angle bars
     ▼
 cv2.imshow
"""

print("Complete pipeline summary:")
print("=" * 60)
stages = [
    ("1. Video capture",     "OpenCV VideoCapture → BGR frames at 30fps"),
    ("2. Pose estimation",   "MediaPipe Pose → 33 landmarks (132 values)"),
    ("3. Feature extraction",f"Joint angles + distances → {N_FEATURES} features/frame"),
    ("4. Exercise selected", "User picks from frontend (no auto-detect needed)"),
    ("5. Rep counting",      "State machine: compares key angle vs thresholds"),
    ("6. Form checking",     "Rule-based biomechanical angle & alignment checks"),
    ("7. Visual feedback",   "HUD overlay with skeleton, reps, form, angles"),
]
for stage, desc in stages:
    print(f"  {stage:25s} → {desc}")
print("=" * 60)
print("\nTo run the full system:")
print("  python main.py")
print("  → Select exercise from the GUI")
print("  → Webcam opens with real-time monitoring")

## Summary

| Step | What | Tool |
|---|---|---|
| Body detection | BlazePose detector finds person ROI | MediaPipe internal |
| Landmark prediction | CNN regresses 33 3D keypoints | MediaPipe internal |
| Feature engineering | Angles + normalized distances | NumPy |
| Classification | Random Forest on per-frame features | scikit-learn |
| Temporal model | LSTM on 30-frame sequences | TensorFlow/Keras |
| Rep counting | State machine on primary joint angle | Pure Python |
| Form feedback | Rule-based biomechanical thresholds | Pure Python |
| Visualization | HUD overlay on OpenCV | OpenCV |

The key insight is that **joint angles** are the most information-rich features because they are:
- **Position invariant** — same squat from anywhere in frame = same angle
- **Scale invariant** — works for tall and short people
- **Biomechanically meaningful** — directly maps to exercise standards
